In [ ]:
import pandas as pd
import numpy as np

ark_final=pd.read_csv('arkansas_integrated.csv')
cal_final=pd.read_csv('california_integrated.csv')
print(ark_final['target_label'])
print(cal_final.columns)

Index(['B2_0', 'B3_0', 'B4_0', 'B5_0', 'B6_0', 'B7_0', 'B8_0', 'B8A_0',
       'B11_0', 'B12_0',
       ...
       'dew_32', 'temp_33', 'precip_33', 'dew_33', 'temp_34', 'precip_34',
       'dew_34', 'temp_35', 'precip_35', 'dew_35'],
      dtype='object', length=513)
Index(['B2_0', 'B3_0', 'B4_0', 'B5_0', 'B6_0', 'B7_0', 'B8_0', 'B8A_0',
       'B11_0', 'B12_0',
       ...
       'dew_32', 'temp_33', 'precip_33', 'dew_33', 'temp_34', 'precip_34',
       'dew_34', 'temp_35', 'precip_35', 'dew_35'],
      dtype='object', length=513)


In [8]:
# Define Feature Groups
# Assumes Sentinel-2 bands are B2_0...B12_35
S2_COLS = [col for col in ark_final.columns if col.startswith(('B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B11', 'B12'))]
CLIMATE_COLS = [col for col in ark_final.columns if col.startswith(('temp', 'precip', 'dew'))]
SOIL_COLS = ['carbon', 'ph','texture'] # Replace with your exact soil column names
TOPO_COLS = ['elevation', 'landforms' ] # Replace with your exact topo column names

def get_xy_split(df, feature_list, target_col='target_label', mask_col='mask'):
    """
    Separates the dataframe into Target, Mask, and Features.
    """
    y = df[target_col].values if target_col in df.columns else None
    mask = df[mask_col].values if mask_col in df.columns else np.ones((len(df), 36)) # Default mask if not present
    X = df[feature_list].values
    
    return X, y, mask

In [9]:
def load_s2_climate(df):
    """Setup 2: Sentinel-2 + Climate Variables"""
    cols = S2_COLS + CLIMATE_COLS
    return get_xy_split(df, cols)

def load_s2_soil(df):
    """Setup 3: Sentinel-2 + Soil Variables"""
    cols = S2_COLS + SOIL_COLS
    return get_xy_split(df, cols)

def load_s2_topography(df):
    """Setup 4: Sentinel-2 + Topography"""
    cols = S2_COLS + TOPO_COLS
    return get_xy_split(df, cols)

def load_s2_all(df):
    """Setup 5: All Covariates Combined"""
    cols = S2_COLS + CLIMATE_COLS + SOIL_COLS + TOPO_COLS
    return get_xy_split(df, cols)

In [10]:
def generate_quality_mask(X_s2):
    """
    Generates a boolean mask where a time-step is 'False' if the 
    Sentinel values are 0 or NaN (common for cloud-cleared pixels).
    """
    # Reshape to (Samples, TimeSteps, Bands) -> (10000, 36, 10)
    X_reshaped = X_s2.reshape(X_s2.shape[0], 36, -1)
    
    # Mask is True if any band in that time step has a valid value (> 0)
    mask = np.any(X_reshaped > 0, axis=2) 
    return mask.astype(np.float32)

# raw data + soil 

In [12]:
X, y, mask = load_s2_soil(ark_final)

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Mask shape: {mask.shape}")

Features shape: (10000, 363)
Target shape: (10000,)
Mask shape: (10000, 36)


# raw data + topography 

In [13]:
X, y, mask = load_s2_topography(ark_final)

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Mask shape: {mask.shape}")

Features shape: (10000, 362)
Target shape: (10000,)
Mask shape: (10000, 36)


# raw data + climate

In [14]:
X, y, mask = load_s2_climate(ark_final)

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Mask shape: {mask.shape}")

Features shape: (10000, 468)
Target shape: (10000,)
Mask shape: (10000, 36)


# raw data + soil + topography + climate

In [15]:
X, y, mask = load_s2_all(ark_final)

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Mask shape: {mask.shape}")

Features shape: (10000, 473)
Target shape: (10000,)
Mask shape: (10000, 36)
